# 18 — Temporal and feature robustness

**Objective.** Run strict-versus-extended feature sensitivity, event-count-matched scarcity envelopes, landmark strata, and secondary/exploratory outcome transport checks without retuning on 2010.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Acquisition remains a secondary different event family; C+36 remains exploratory when event gates fail.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("18", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json
import numpy as np
import pandas as pd
from cruxvc.calibration import CalibratedModel, PlattCalibrator
from cruxvc.explanations import active_feature_groups, explain_dataset, load_feature_groups
from cruxvc.io import read_table, write_table
from cruxvc.manifest import append_test_access_log
from cruxvc.metrics import attribution_distance_bundle, binary_prediction_metrics
from cruxvc.models import fit_model, predict_positive

append_test_access_log(P, stage_id="18", purpose="locked feature/temporal robustness", resources=[P.processed / "features_extended.parquet", P.processed / "cohort_labels.parquet"])
strict = read_table(P.processed / "features_strict.parquet")
extended = read_table(P.processed / "features_extended.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
splits = read_table(P.protocol / "split_ids.parquet")
reference = read_table(P.models / "matched_reference_configs.parquet").sort_values("platt_oof_log_loss").iloc[0]
params = json.loads(reference["parameters_json"])


In [ ]:
robustness_rows = []
fitted_predictions = []
calibrated_models = {}
feature_frames = {}
feature_columns_by_set = {}
for feature_set_name, features in [("strict", strict), ("extended", extended)]:
    frame = features.merge(cohort, on=["case_id", "company_permalink", "t0"]).merge(splits[["case_id", "time_block"]], on="case_id")
    development = frame[frame["time_block"].astype(str).eq("development")].copy()
    cal2008 = frame[frame["time_block"].astype(str).eq("probability_calibration")].copy()
    test = frame[frame["time_block"].astype(str).eq("final_test")].copy()
    feature_columns = [c for c in features.columns if c not in {"case_id", "company_permalink", "t0"}]
    feature_frames[feature_set_name] = {"development": development, "cal2008": cal2008, "test": test}
    feature_columns_by_set[feature_set_name] = feature_columns
    for outcome in ["F18", "F36", "B+36", "A36", "C+36"]:
        model = fit_model(development, development[outcome], feature_columns, reference["family"], params, int(reference["seed"]))
        raw_cal = predict_positive(model, cal2008, feature_columns)
        calibrator = PlattCalibrator().fit(raw_cal, cal2008[outcome])
        calibrated = CalibratedModel(model, calibrator, tuple(feature_columns))
        calibrated_models[(feature_set_name, outcome)] = calibrated
        probability = calibrated.predict_proba(test)[:, 1]
        metric = binary_prediction_metrics(test[outcome], probability)
        robustness_rows.append({"analysis": "feature_set", "feature_set": feature_set_name, "outcome": outcome, **metric})
        fitted_predictions.append(pd.DataFrame({"case_id": test["case_id"], "feature_set": feature_set_name, "outcome": outcome, "probability": probability, "y_true": test[outcome]}))


In [ ]:
# Development-only event-count matching: downsample the more prevalent positive outcome to the rarer count; negatives remain.
scarcity_rows = []
rng = np.random.default_rng(int(CFG["execution"]["random_seed"]) + 181)
strict_frame = strict.merge(cohort, on=["case_id", "company_permalink", "t0"]).merge(splits[["case_id", "time_block"]], on="case_id")
dev = strict_frame[strict_frame["time_block"].astype(str).eq("development")]
cal = strict_frame[strict_frame["time_block"].astype(str).eq("probability_calibration")]
test = strict_frame[strict_frame["time_block"].astype(str).eq("final_test")]
feature_columns = [c for c in strict.columns if c not in {"case_id", "company_permalink", "t0"}]
for left, right in [("F18", "F36"), ("F36", "B+36")]:
    target_events = min(int(dev[left].sum()), int(dev[right].sum()))
    for outcome in [left, right]:
        positives = dev[dev[outcome].eq(1)]
        negatives = dev[dev[outcome].eq(0)]
        chosen = rng.choice(positives.index, target_events, replace=False)
        matched_dev = pd.concat([negatives, positives.loc[chosen]]).sort_values("case_id")
        model = fit_model(matched_dev, matched_dev[outcome], feature_columns, reference["family"], params, int(reference["seed"]))
        calibrator = PlattCalibrator().fit(predict_positive(model, cal, feature_columns), cal[outcome])
        calibrated = CalibratedModel(model, calibrator, tuple(feature_columns))
        probability = calibrated.predict_proba(test)[:, 1]
        scarcity_rows.append({"contrast": f"{left}_vs_{right}", "outcome": outcome, "matched_development_positive_n": target_events, **binary_prediction_metrics(test[outcome], probability)})

In [ ]:
# Strict-versus-extended explanation sensitivity uses the same cases,
# background IDs, outcome-specific model parameters, and approximation budget.
all_groups = load_feature_groups(P.config / "feature_groups.yaml")
audit_ids = pd.read_csv(P.protocol / "local_audit_case_ids.csv")["case_id"]
background_registry = pd.read_csv(P.protocol / "explanation_background_ids.csv")
background_id = sorted(background_registry["background_id"].unique())[0]
background_case_ids = (
    background_registry[background_registry["background_id"].eq(background_id)]
    .sort_values("order")["case_id"]
    .head(int(PROFILE["background_n"]))
)
explanation_parts = []
for feature_set_name in ["strict", "extended"]:
    feature_columns = feature_columns_by_set[feature_set_name]
    groups = active_feature_groups(all_groups, feature_columns)
    development = feature_frames[feature_set_name]["development"]
    test = feature_frames[feature_set_name]["test"]
    background = development[development["case_id"].isin(background_case_ids)].copy()
    cases = test[test["case_id"].isin(audit_ids)].sort_values("case_id").copy()
    if len(background) == 0 or len(cases) == 0:
        raise RuntimeError(f"Missing frozen explanation cases/background for {feature_set_name}")
    for outcome in CFG["outcomes"]["confirmatory"]:
        explanation_parts.append(explain_dataset(
            calibrated_models[(feature_set_name, outcome)],
            cases,
            background,
            groups,
            feature_columns,
            model_metadata={
                "feature_set": feature_set_name,
                "outcome": outcome,
                "family": reference["family"],
                "config_id": reference["config_id"],
                "model_id": f"{feature_set_name}_{outcome}_robustness",
                "refit_id": "feature_set_sensitivity",
            },
            background_id=background_id,
            approximation_seed=int(CFG["execution"]["random_seed"]) + 180,
            n_orderings=int(PROFILE["permutation_orderings"]),
        ))
feature_attributions = pd.concat(explanation_parts, ignore_index=True)
mean_attr = (
    feature_attributions.groupby(["case_id", "outcome", "feature_set", "feature_group"], as_index=False)["phi_log_odds"]
    .mean()
)
wide = mean_attr.pivot_table(
    index=["case_id", "outcome"], columns=["feature_set", "feature_group"], values="phi_log_odds", aggfunc="mean"
)
feature_groups_union = sorted(mean_attr["feature_group"].unique())
explanation_sensitivity_rows = []
for (case_id, outcome), row in wide.iterrows():
    strict_vector = np.array([row.get(("strict", group), 0.0) for group in feature_groups_union], dtype=float)
    extended_vector = np.array([row.get(("extended", group), 0.0) for group in feature_groups_union], dtype=float)
    strict_vector = np.nan_to_num(strict_vector, nan=0.0)
    extended_vector = np.nan_to_num(extended_vector, nan=0.0)
    explanation_sensitivity_rows.append({
        "case_id": int(case_id),
        "outcome": outcome,
        "family": reference["family"],
        "config_id": reference["config_id"],
        "background_id": background_id,
        "approximation_seed": int(CFG["execution"]["random_seed"]) + 180,
        "strict_total_absolute_mass": float(np.abs(strict_vector).sum()),
        "extended_total_absolute_mass": float(np.abs(extended_vector).sum()),
        **attribution_distance_bundle(strict_vector, extended_vector, feature_groups_union),
    })
explanation_sensitivity = pd.DataFrame(explanation_sensitivity_rows)


In [ ]:
final_predictions = read_table(P.predictions / "final_predictions.parquet")
metadata = cohort[["case_id", "landmark_round_type"]]
strata_rows = []
for (outcome, family, config_id), group in final_predictions[final_predictions["analysis_role"].eq("matched_reference_deployment")].merge(metadata, on="case_id").groupby(["outcome", "family", "config_id"]):
    for landmark, part in group.groupby("landmark_round_type"):
        strata_rows.append({"outcome": outcome, "family": family, "config_id": config_id, "landmark_round_type": landmark, **binary_prediction_metrics(part["y_true"], part["probability"])})
feature_results = pd.DataFrame(robustness_rows)
scarcity = pd.DataFrame(scarcity_rows)
strata = pd.DataFrame(strata_rows)
feature_path = write_table(feature_results, P.inference / "strict_extended_feature_robustness.csv")
prediction_path = write_table(pd.concat(fitted_predictions, ignore_index=True), P.predictions / "strict_extended_test_predictions.parquet")
scarcity_path = write_table(scarcity, P.inference / "event_count_matched_scarcity_envelope.csv")
strata_path = write_table(strata, P.inference / "landmark_strata_performance.csv")
explanation_path = write_table(explanation_sensitivity, P.inference / "strict_extended_explanation_sensitivity.csv")
CTX.recorder.complete([feature_path, prediction_path, scarcity_path, strata_path, explanation_path])
print(feature_results[["feature_set", "outcome", "average_precision", "brier_skill"]].to_string(index=False))
print(explanation_sensitivity.groupby("outcome")[["sqrt_jsd", "top_k_jaccard", "sign_agreement"]].mean().to_string())
